In [0]:
from pyspark.sql.functions import current_timestamp, col
import json


dbutils.widgets.text("catalog", "dbr_dev")
dbutils.widgets.text("schema", "janvander0912_bronze")
dbutils.widgets.text("storage_account", "dlspl21databricks")
dbutils.widgets.text("container", "janvander0912")
dbutils.widgets.text("volume", "raw_data")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
storage_account = dbutils.widgets.get("storage_account")
container = dbutils.widgets.get("container")
volume = dbutils.widgets.get("volume")


base_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/{volume}/"
input_path = f"{base_path}truck_stream/"

# checkpoint_path = f"{base_path}_checkpoints/trucks_v1"
# schema_path = f"{base_path}_schemas/trucks_v1/"
checkpoint_path = f"{base_path}_checkpoints/trucks_v2"
schema_path = f"{base_path}_schemas/trucks_v2/"

table_name = f"{catalog}.{schema}.truck_cold_chain_bronze"

In [0]:
print(f"Stream configuration for table: {table_name}")
df_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_path)         # 
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")  # 
    .option("cloudFiles.inferColumnTypes", "true")            # 
    .load(input_path)

    #  Transformations and metadata columns
    .select(
        "*",
        col("_metadata.file_name").alias("source_filename")   # 
    )
    .withColumn("ingestion_timestamp", current_timestamp())
)

In [0]:
query = (df_stream.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")     
    .trigger(availableNow=True)
    .toTable(table_name)        
)

print("Activating stream...")
query.awaitTermination()
print("Success! Stream finnished (all files were processed)")


if query.recentProgress:
    last_batch = query.recentProgress[-1]
    print(f"\n Batch ID: {last_batch.get('batchId')}")
    print(f"Number of records loaded: {last_batch.get('numInputRows')}")
    print(json.dumps(last_batch.get('sources')[0].get('metrics'), indent=2))

display(spark.sql(f"""
    SELECT measurement_id, cargo_type, current_temp, status, door_open, _rescued_data 
    FROM {table_name} 
    ORDER BY measurement_id DESC 
    LIMIT 15
"""))